# SWEAチャットボット v3
### スライド窓アンサンブル法（N乗融合 + KVキャッシュ）
- 会話が短い間は通常生成（速い）
- 会話が長くなったら自動でN乗融合アンサンブルに切り替え
- **KVキャッシュ**：SINK+WINDOW部分を再利用し、2ステップ目以降はLOCALだけ計算
- GradioのスライダーでENSEMBLE_Nをリアルタイム調整可能

In [ ]:
!pip install transformers torch accelerate gradio -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("モデルを読み込んでいます...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("完了")


In [ ]:
# ===== 設定 =====
SINK_SIZE      = 10
WINDOW_SIZE    = 512
LOCAL_SIZE     = 100
BATCH_SIZE     = 16
MIN_TOKENS     = SINK_SIZE + WINDOW_SIZE + LOCAL_SIZE  # 622
MAX_NEW_TOKENS = 200
ENSEMBLE_N     = 2.0  # N乗融合の乗数（Gradioスライダーで上書き可）

SYSTEM_PROMPT = "あなたは親切なアシスタントです。日本語で簡潔に答えてください。"

print(f"設定完了")
print(f"アンサンブル切り替え閾値: {MIN_TOKENS} トークン")
print(f"N乗融合の乗数: {ENSEMBLE_N}")


In [ ]:
import torch.nn.functional as F

class SWEAWithCache:
    def __init__(self, model, tokenizer):
        self.model     = model
        self.tokenizer = tokenizer
        # (w_start, w_end) -> past_key_values（SINK+WINDOW部分）
        self.kv_cache  = {}

    # ------------------------------------------------------------------
    # 公開メソッド
    # ------------------------------------------------------------------
    def predict_next_logits(self, input_ids, n=None):
        """
        次トークンの予測を返す。
        トークン数 < MIN_TOKENS → 通常生成（Logits）
        トークン数 >= MIN_TOKENS → N乗融合アンサンブル + KVキャッシュ（正規化確率）
        """
        _n        = n if n is not None else ENSEMBLE_N
        total_len = input_ids.shape[1]
        mid_end   = total_len - LOCAL_SIZE

        # ---- 通常生成 ----
        if total_len < MIN_TOKENS:
            with torch.no_grad():
                out = self.model(input_ids)
            return out.logits[:, -1, :], "通常生成"

        # ---- アンサンブル + KVキャッシュ ----
        windows   = self._build_windows(total_len)
        all_probs = []

        for w_start, w_end in windows:
            logits = self._predict_window(input_ids, w_start, w_end, mid_end)
            probs  = F.softmax(logits, dim=-1)
            all_probs.append(probs ** _n)

        fused = torch.cat(all_probs, dim=0).mean(dim=0, keepdim=True)
        fused = fused / fused.sum(dim=-1, keepdim=True)
        return fused, f"アンサンブル+KVキャッシュ（窓数{len(windows)}, N={_n}）"

    def clear_cache(self):
        """会話リセット時にKVキャッシュを破棄"""
        self.kv_cache.clear()

    # ------------------------------------------------------------------
    # 内部メソッド
    # ------------------------------------------------------------------
    def _build_windows(self, total_len):
        stride  = WINDOW_SIZE // 2
        mid_end = total_len - LOCAL_SIZE
        windows = []
        pos = SINK_SIZE
        while pos + WINDOW_SIZE <= mid_end:
            windows.append((pos, pos + WINDOW_SIZE))
            pos += stride
        last_start = mid_end - WINDOW_SIZE
        last_end   = mid_end
        if last_start >= SINK_SIZE:
            if len(windows) == 0 or windows[-1] != (last_start, last_end):
                windows.append((last_start, last_end))
        return windows

    def _predict_window(self, input_ids, w_start, w_end, mid_end):
        """
        1窓分の推論。
        SINK+WINDOW は KVキャッシュを再利用し、LOCAL だけ毎回計算する。

        キャッシュの有効条件：
          - (w_start, w_end) が同じ → SINK+WINDOW のトークン列が同じ
          - mid_end（= total_len - LOCAL_SIZE）は可変だが、
            SINK+WINDOW 範囲は変わらないのでキャッシュは使い回せる
        """
        window_key = (w_start, w_end)

        if window_key not in self.kv_cache:
            # ---- キャッシュミス：SINK+WINDOW をフル計算してKVを保存 ----
            sink_window_ids = torch.cat([
                input_ids[:, :SINK_SIZE],
                input_ids[:, w_start:w_end]
            ], dim=1)

            with torch.no_grad():
                out_sw = self.model(sink_window_ids, use_cache=True)

            # KVキャッシュを保存（LOCAL が変わっても再利用可能）
            self.kv_cache[window_key] = out_sw.past_key_values

        # ---- LOCAL だけ追加で流す（毎回計算が必要） ----
        local_ids = input_ids[:, mid_end:]

        with torch.no_grad():
            out = self.model(
                local_ids,
                past_key_values=self.kv_cache[window_key],
                use_cache=False  # LOCALのKVは保存不要
            )

        return out.logits[:, -1, :]


# ------------------------------------------------------------------
# チャット用ユーティリティ
# ------------------------------------------------------------------
swea = SWEAWithCache(model, tokenizer)


def build_prompt(history, user_message):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for human, assistant in history:
        messages.append({"role": "user",      "content": human})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": user_message})
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_response(history, user_message, ensemble_n=None):
    # 新しい会話ターンごとにキャッシュをリセット
    # （前のターンとWINDOW位置がずれるため）
    swea.clear_cache()

    prompt      = build_prompt(history, user_message)
    input_ids   = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    generated   = input_ids.clone()
    token_count = input_ids.shape[1]
    mode_log    = ""

    for _ in range(MAX_NEW_TOKENS):
        out, mode = swea.predict_next_logits(generated, n=ensemble_n)
        if not mode_log:
            mode_log = mode

        next_token_id = out.argmax(dim=-1, keepdim=True)
        if next_token_id.item() == tokenizer.eos_token_id:
            break
        generated = torch.cat([generated, next_token_id], dim=1)

    response   = tokenizer.decode(
        generated[0, input_ids.shape[1]:],
        skip_special_tokens=True
    )
    debug_info = f"\n\n---\nトークン数: {token_count} | モード: {mode_log}"
    return response + debug_info


print("SWEAWithCache の定義完了")


In [ ]:
import gradio as gr

def chat(message, history, ensemble_n):
    return generate_response(history, message, ensemble_n=ensemble_n)


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# SWEAチャットボット v3")
    gr.Markdown(
        "スライド窓アンサンブル法（N乗融合 + KVキャッシュ）による長文対応チャットボット  \n"
        "会話が622トークンを超えると自動でアンサンブルモードに切り替わります  \n"
        "**KVキャッシュ**により2ステップ目以降はLOCAL部分だけ計算（高速化）"
    )

    with gr.Row():
        n_slider = gr.Slider(
            minimum=1.0, maximum=10.0, value=2.0, step=0.5,
            label="ENSEMBLE_N（1.0=通常平均　大きいほど確信度重視）"
        )

    gr.ChatInterface(
        fn=chat,
        additional_inputs=[n_slider],
        examples=[
            ["こんにちは！あなたは何ができますか？"],
            ["Pythonでフィボナッチ数列を計算するコードを書いてください"],
            ["東京のおすすめ観光地を教えてください"],
        ],
    )

demo.launch(share=True)
